# 09 Results: every model on identical test answers (C1 Adaptive Curriculum Engine)

One table for the report. BKT, DKT and GKT predictions come from notebook 05 (`baselines/test_predictions.parquet`); `gat-v2` is scored here with the exported model on **exactly the same answers**, matched by (student, step): answers 2 to 200 of each 200-answer chunk of the 6,000 held-out test students.

| Statistic | How |
|---|---|
| AUC (primary), accuracy, RMSE | on the identical answers |
| Majority class | always predict "correct" (accuracy = correct rate) |
| 95% interval of the AUC difference | bootstrap over **students**, not answers (a student's answers are not independent) |
| Paired t-test | AUC in 10 fixed groups of test students, each model vs gat-v2 |

Needs notebooks 01 to 05 (05 re-run after 11 Oct, so it saves its predictions). CPU is enough.

In [1]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 1,000 test students, for a fast check.
QUICK = False

In [2]:
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn networkx scipy

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for needed in ('interactions.parquet', 'splits.json', 'bkt/params.csv', 'graph/edge_index.npy', 'gat/feature_spec.json', 'baselines/test_predictions.parquet'):
    assert (PROCESSED / needed).exists(), f"{needed} missing: run the earlier notebooks first."
print("Storage:", STORAGE, "->", BASE)

Mounted at /content/drive
Storage: drive -> /content/drive/MyDrive/adaptlearn-c1


## 1. Load the baseline predictions, the answers and gat-v2

In [3]:
import hashlib
import json
import math
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

SEED = 42
rng = np.random.default_rng(SEED)

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise_idx", "step", "correct"])
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
names = sorted(exercise_index, key=exercise_index.get)
n_ex = len(names)
edge_index = np.load(PROCESSED / "graph" / "edge_index.npy")  # row 0 prerequisite, row 1 exercise
bkt = pd.read_csv(PROCESSED / "bkt" / "params.csv").set_index("exercise").loc[names]

# gat-v2 exactly as exported by notebook 04 (and run by curriculum-service).
spec = json.loads((PROCESSED / "gat" / "feature_spec.json").read_text())
model_path = PROCESSED / "gat" / f"{spec['model_version']}.npz"
model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
assert model_sha == spec["sha256"], "gat model file does not match its feature spec"
weights = dict(np.load(model_path))
ARCH = spec["architecture"]
print(f"{spec['model_version']} sha256 {model_sha}")

G = nx.DiGraph()
G.add_nodes_from(range(n_ex))
G.add_edges_from(zip(edge_index[0].tolist(), edge_index[1].tolist()))
assert nx.is_directed_acyclic_graph(G)
depth = np.zeros(n_ex)
for v in nx.topological_sort(G):
    preds = list(G.predecessors(v))
    depth[v] = 1 + max(depth[p] for p in preds) if preds else 0
prereqs = [sorted(G.predecessors(v)) for v in range(n_ex)]
K = max(len(p) for p in prereqs)

gat-v2 sha256 914a3046ac95daa2a58228de2ac6e75811c83f625760c826594123bf2c9cb17d


In [4]:
CONFIG = {"SEED": SEED, "QUICK": QUICK, "BOOTSTRAP": 200, "GROUPS": 10}
base = pd.read_parquet(PROCESSED / "baselines" / "test_predictions.parquet")
baseline_summary = json.loads((PROCESSED / "baselines" / "summary.json").read_text())
assert not baseline_summary.get("quick", False) or QUICK, "notebook 05 ran in QUICK mode: re-run it in full"
students = sorted(base.user_id.unique())
if QUICK:
    students = sorted(rng.choice(students, min(1_000, len(students)), replace=False).tolist())
    base = base[base.user_id.isin(set(students))]
assert set(students) <= set(splits["test"]), "baseline rows include non-test students"
test = inter[inter.user_id.isin(set(students))].reset_index(drop=True)
print(f"{len(base):,} scored answers from {len(students):,} test students "
      f"({len(test):,} answers in their full histories)")

1,168,144 scored answers from 6,000 test students (1,178,267 answers in their full histories)


## 2. gat-v2 on the full history of each test student

In [5]:
# Notebook 04's features, from answers before t only, over each student's whole history.
FEATURES = spec["features"]
LEARNER = spec["learner_features"]
assert FEATURES == ["mastery", "log_evidence", "log_depth", "log_prereq_count",
                    "mean_prereq_mastery", "bkt_p_correct"]
assert LEARNER == ["overall_correct_rate", "recent_correct_rate", "previous_correct"]
BKT_P = FEATURES.index("bkt_p_correct")
F, NL, RECENT = len(FEATURES), len(LEARNER), 5
P_INIT, P_LEARN, P_GUESS, P_SLIP = (bkt[c].to_numpy().tolist()
                                    for c in ("p_init", "p_learn", "p_guess", "p_slip"))
LOG_DEPTH = np.log1p(depth).tolist()
LOG_PREREQS = [math.log1p(len(p)) for p in prereqs]


def node_features(v, state, count):
    ps = prereqs[v]
    mean_pm = sum(state[p] for p in ps) / len(ps) if ps else 1.0
    m = state[v]
    return (m, math.log1p(count[v]), LOG_DEPTH[v], LOG_PREREQS[v], mean_pm,
            m * (1 - P_SLIP[v]) + (1 - m) * P_GUESS[v])


def build_features(frame):
    """X_t [n,F], X_n [n,K,F], M [n,K], X_l [n,NL] for every row of `frame` (sorted by user, step)."""
    n = len(frame)
    X_t = np.zeros((n, F), np.float32)
    X_n = np.zeros((n, K, F), np.float32)
    M = np.zeros((n, K), bool)
    X_l = np.zeros((n, NL), np.float32)
    users = frame.user_id.to_numpy()
    ex = frame.exercise_idx.to_numpy().tolist()
    y = frame.correct.to_numpy().astype(bool).tolist()
    new_student = np.r_[True, users[1:] != users[:-1]].tolist()
    for i in tqdm(range(n), mininterval=5):
        if new_student[i]:
            state, count = list(P_INIT), [0] * n_ex
            answered, right, recent, previous = 0, 0, deque(maxlen=RECENT), 0.5
        X_l[i] = ((right + 1) / (answered + 2), (sum(recent) + 1) / (len(recent) + 2), previous)
        v = ex[i]
        X_t[i] = node_features(v, state, count)
        for j, u in enumerate(prereqs[v]):
            X_n[i, j] = node_features(u, state, count)
            M[i, j] = True
        m, g, s = state[v], P_GUESS[v], P_SLIP[v]
        if y[i]:
            post = m * (1 - s) / (m * (1 - s) + (1 - m) * g)
        else:
            post = m * s / (m * s + (1 - m) * (1 - g))
        state[v] = post + (1 - post) * P_LEARN[v]
        count[v] += 1
        answered += 1
        right += y[i]
        recent.append(y[i])
        previous = float(y[i])
    return X_t, X_n, M, X_l


def gat_forward_numpy(w, x_t, x_n, mask, x_l, heads, hidden, slope):
    """Notebook 04's reference forward pass (also copied into curriculum-service)."""
    B, k, _ = x_n.shape
    nodes = np.concatenate([x_t[:, None], x_n], 1)
    h = (nodes @ w["W"]).reshape(B, 1 + k, heads, hidden)
    score = (h[:, :1] * w["a_dst"]).sum(-1) + (h * w["a_src"]).sum(-1)
    score = np.where(score > 0, score, slope * score)
    present = np.concatenate([np.ones((B, 1), bool), mask], 1)
    score = np.where(present[..., None], score, -np.inf)
    score = score - score.max(1, keepdims=True)
    alpha = np.exp(score) / np.exp(score).sum(1, keepdims=True)
    z = (alpha[..., None] * h).sum(1).reshape(B, heads * hidden)
    z = np.where(z > 0, z, np.expm1(np.minimum(z, 0)))
    hidden1 = np.maximum(np.concatenate([z, x_t, x_l], -1) @ w["fc1_w"] + w["fc1_b"], 0)
    p_bkt = np.clip(x_t[:, BKT_P], 1e-4, 1 - 1e-4)
    logit = np.log(p_bkt / (1 - p_bkt)) + (hidden1 @ w["fc2_w"] + w["fc2_b"])[:, 0]
    return 1 / (1 + np.exp(-logit)), alpha


def gat(x_t, x_n, mask, x_l, batch=200_000):
    out_p, out_a = [], []
    for b in range(0, len(x_t), batch):
        s = slice(b, b + batch)
        p, a = gat_forward_numpy(weights, x_t[s], x_n[s], mask[s], x_l[s],
                                 ARCH["heads"], ARCH["hidden"], ARCH["leaky_slope"])
        out_p.append(p)
        out_a.append(a)
    return np.concatenate(out_p), np.concatenate(out_a)

In [6]:
X_t, X_n, M, X_l = build_features(test)
test["p_gat"] = gat(X_t, X_n, M, X_l)[0]
df = base.merge(test[["user_id", "step", "correct", "p_gat"]], on=["user_id", "step"],
                how="left", suffixes=("", "_inter"), validate="one_to_one")
assert df.p_gat.notna().all(), "some baseline answers have no gat-v2 prediction"
assert (df.correct == df.correct_inter).all(), "labels differ: different answers were matched"
df = df.drop(columns="correct_inter").reset_index(drop=True)
MODELS = {"BKT": "p_bkt", "DKT": "p_dkt", "GKT": "p_gkt", "BKT + GAT (gat-v2)": "p_gat"}
print(f"{len(df):,} answers, every model present on every one")

  0%|          | 0/1178267 [00:00<?, ?it/s]

1,168,144 answers, every model present on every one


## 3. The results table

In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score

y = df.correct.to_numpy().astype(bool)
table = [{"model": "majority class (always correct)", "auc": 0.5, "accuracy": y.mean(),
          "rmse": float(np.sqrt(((y - y.mean()) ** 2).mean()))}]
for name, col in MODELS.items():
    p = df[col].to_numpy()
    table.append({"model": name, "auc": roc_auc_score(y, p), "accuracy": accuracy_score(y, p >= 0.5),
                  "rmse": float(np.sqrt(((y - p) ** 2).mean()))})
table = pd.DataFrame(table)
table.round(4)

,model,auc,accuracy,rmse
0,majority class (always correct),0.5000,0.8348,0.3713
1,BKT,0.7804,0.8516,0.3354
2,DKT,0.7877,0.8519,0.3343
3,GKT,0.7885,0.8528,0.3337
4,BKT + GAT (gat-v2),0.7931,0.8550,0.3315


## 4. Is gat-v2 different from each baseline?

Student bootstrap: resample the test students with replacement, recompute both AUCs on the same resampled answers, keep the difference. Paired t-test: split the students into 10 fixed groups (seed 42), compute each model's AUC per group, compare with `ttest_rel`.

In [8]:
from scipy.stats import ttest_rel

owner = df.user_id.to_numpy()
uniq, inverse = np.unique(owner, return_inverse=True)
rows_of = [np.flatnonzero(inverse == k) for k in range(len(uniq))]
boot_rng = np.random.default_rng(SEED)
others = [m for m in MODELS if m != "BKT + GAT (gat-v2)"]
diffs = {m: [] for m in others}
for _ in tqdm(range(CONFIG["BOOTSTRAP"])):
    take = np.concatenate([rows_of[k] for k in boot_rng.integers(0, len(uniq), len(uniq))])
    gat_auc = roc_auc_score(y[take], df.p_gat.to_numpy()[take])
    for m in others:
        diffs[m].append(gat_auc - roc_auc_score(y[take], df[MODELS[m]].to_numpy()[take]))

group_of = dict(zip(uniq, np.random.default_rng(SEED).permutation(len(uniq)) % CONFIG["GROUPS"]))
groups = np.array([group_of[u] for u in owner])
per_group = pd.DataFrame({name: [roc_auc_score(y[groups == g], df[col].to_numpy()[groups == g])
                                 for g in range(CONFIG["GROUPS"])]
                          for name, col in MODELS.items()})
comparison = []
for m in others:
    t, p = ttest_rel(per_group["BKT + GAT (gat-v2)"], per_group[m])
    lo, hi = np.percentile(diffs[m], [2.5, 97.5])
    comparison.append({"gat-v2 minus": m,
                       "auc_difference": table.set_index("model").auc["BKT + GAT (gat-v2)"]
                                         - table.set_index("model").auc[m],
                       "ci95_low": lo, "ci95_high": hi,
                       "groups_gat_better": int((per_group["BKT + GAT (gat-v2)"] > per_group[m]).sum()),
                       "paired_t": t, "p_value": p})
comparison = pd.DataFrame(comparison)
display(comparison.round(4))
display(per_group.round(4))

  0%|          | 0/200 [00:00<?, ?it/s]

,gat-v2 minus,auc_difference,ci95_low,ci95_high,groups_gat_better,paired_t,p_value
0,BKT,0.0127,0.0115,0.0143,10,16.5705,0.0000
1,DKT,0.0054,0.0040,0.0076,10,5.2536,0.0005
2,GKT,0.0046,0.0033,0.0060,10,5.9539,0.0002


,BKT,DKT,GKT,BKT + GAT (gat-v2)
0,0.7770,0.7885,0.7855,0.7932
1,0.7763,0.7860,0.7865,0.7896
2,0.7824,0.7879,0.7897,0.7918
3,0.7783,0.7837,0.7858,0.7903
4,0.7826,0.7849,0.7893,0.7992
5,0.7822,0.7908,0.7894,0.7955
6,0.7936,0.8005,0.8007,0.8033
7,0.7837,0.7901,0.7913,0.7946
8,0.7766,0.7832,0.7854,0.7884
9,0.7707,0.7805,0.7794,0.7843


## 5. Save results and a manifest

In [9]:
table.to_csv(OUT_DIR / "results_table.csv", index=False)
comparison.to_csv(OUT_DIR / "gat_vs_baselines.csv", index=False)
per_group.to_csv(OUT_DIR / "auc_per_student_group.csv", index=False)
_ = (OUT_DIR / "summary.json").write_text(json.dumps({
    "answers": int(len(df)), "students": int(len(uniq)), "correct_rate": float(y.mean()),
    "scored_rows": baseline_summary["scored_rows"],
}, indent=2))

In [10]:
import platform, scipy, sklearn
from datetime import datetime, timezone

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "model": {"version": spec["model_version"], "sha256": model_sha},
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scipy": scipy.__version__,
                 "scikit-learn": sklearn.__version__, "networkx": nx.__version__},
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive(f"/content/{OUT_DIR.name}", "zip", OUT_DIR))

{
  "created_at": "2026-10-11T15:05:55.544491+00:00",
  "config": {
    "SEED": 42,
    "QUICK": false,
    "BOOTSTRAP": 200,
    "GROUPS": 10
  },
  "model": {
    "version": "gat-v2",
    "sha256": "914a3046ac95daa2a58228de2ac6e75811c83f625760c826594123bf2c9cb17d"
  },
  "versions": {
    "python": "3.13.15",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "networkx": "3.7"
  },
  "sha256": {
    "auc_per_student_group.csv": "80ca7711563c41ce43e752cef52a9e18f511c6db12119b6e8cb7c44db39bafc7",
    "gat_vs_baselines.csv": "2cda2d01bfaeef171bfab976d966b2df7bcce2e70db4a72f5bc66275213dce33",
    "results_table.csv": "be6d87e4a91369946bd019db20ad5f1d359a7cc897b77078a9da80cc0219c784",
    "summary.json": "a63c0c7c2fba3fc4327460580520ab96b28d7ad418d1f54eb0ced833838861d0"
  }
}


## For the report

Use `results_table.csv` as the main results table (AUC primary; accuracy against the majority class) and `gat_vs_baselines.csv` for the significance statements.

- The gat-v2 AUC here is on answers 2 to 200 of each chunk, so it can differ slightly from notebook 04's 0.7930 (every test answer). Quote this notebook's number when comparing models.
- Read a difference as significant only when the 95% bootstrap interval excludes 0 **and** the paired t-test gives p < 0.05.
- Whatever the ranking against DKT and GKT, state the trade-off: gat-v2 uses only graph-agnostic features, so it transfers to the Java PF and DSA graph, and its attention gives the explanation (notebook 07); DKT and GKT learn per-exercise embeddings that do not exist for the Java concepts.

Before committing: **Clear all outputs**.